# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² Kilifi mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset high-level metadata
md = dataset.metadata  # not subscripted! Treat as an object, not a dict.
print(f"{md.name}: {md.description}\n")
print(f"Version: {md.version}")
print(f"Published: {getattr(md, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(md, 'keywords', 'N/A')}")
print(f"License: {getattr(md, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs. All references in this notebook are by `@id` as required for Croissant datasets.

In [ ]:
# List all record sets and their `@id`s, and their fields.
# mlcroissant stores record set ids in the Dataset object.
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"- {rs}")
    record_set_metadata = dataset.record_set_metadata(rs)
    fields = getattr(record_set_metadata, 'fields', [])
    col_ids = [getattr(f, '@id', str(f)) for f in fields]
    print("  Fields (@id): ", col_ids)
    columns = getattr(record_set_metadata, 'columns', [])
    col_col_ids = [getattr(c, '@id', str(c)) for c in columns]
    if col_col_ids:
        print("  Columns (@id):", col_col_ids)
    print()

## 3. Data Extraction
Load all records from available record sets into DataFrames for analysis. Use the record set and field/column `@id`s identified above.

In [ ]:
# Load all record sets into pandas DataFrames
dataframes = {}
for record_set_id in dataset.record_sets:
    print(f"Loading records for {record_set_id} ...")
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set_id] = df
    print(f"{record_set_id} rows: {df.shape[0]}, columns: {df.shape[1]}")
    print(f"Columns (@id): {list(df.columns)}\n")

Show data sample for the main survey record set. The record set containing the main survey data typically has fields like GAD-7, PHQ-9, PSQ, and demographic data. Referenced by its `@id` as found above.


In [ ]:
# Select the main survey record set (please update `main_record_set_id` if needed based on the previous cell's output)

# Example: assume only one main record set, or pick one by @id
main_record_set_id = dataset.record_sets[0]
main_df = dataframes[main_record_set_id]
print(f"First 5 rows of `{main_record_set_id}` record set:")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This sample applies EDA to a numeric field, e.g., GAD-7 or PHQ-9 scores, by referencing the correct field `@id` as seen above.

In [ ]:
# Specify field @id for GAD-7, PHQ-9, or PSQ score as found above.
# You may want to adjust `score_field_id` and `group_field_id` based on available fields.
score_field_id = None
group_field_id = None

# Try to find likely candidates for score and group fields
for col in main_df.columns:
    if ('gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()) and 'score' in col.lower():
        score_field_id = col
    if ('gender' in col.lower() or 'sex' in col.lower()) and group_field_id is None:
        group_field_id = col
if score_field_id is None:
    print('No GAD-7/PHQ-9/PSQ score field found, please check columns.')
else:
    print(f"Analyzing score field: {score_field_id}")

    # Drop NA for score field
    filtered_df = main_df.dropna(subset=[score_field_id])

    # Convert to numeric if needed
    filtered_df[score_field_id] = pd.to_numeric(filtered_df[score_field_id], errors='coerce')

    # Filter records with score above a threshold (e.g. 10)
    threshold = 10
    filtered = filtered_df[filtered_df[score_field_id] > threshold]
    print(f"Records with {score_field_id} > {threshold}: {len(filtered)}")
    print(filtered.head())

    # Normalize the score field (z-score)
    norm_col = f"{score_field_id}_normalized"
    filtered[norm_col] = (filtered[score_field_id] - filtered[score_field_id].mean()) / filtered[score_field_id].std()
    print(f"\nNormalized score field '{score_field_id}':")
    print(filtered[[score_field_id, norm_col]].head())

    # Optionally group by gender/sex or other demographic column
    if group_field_id and group_field_id in filtered.columns:
        print(f"\nGrouped mean scores by '{group_field_id}':")
        grouped = filtered.groupby(group_field_id)[score_field_id].mean().reset_index()
        print(grouped.head())

## 5. Visualization
Visualize data distributions or field relationships. This example plots the score field distribution and optionally score vs. demographic group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if score_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[score_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {score_field_id}")
    plt.xlabel(score_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[score_field_id])
        plt.title(f"{score_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(score_field_id)
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load and explore the FAIR² Kilifi mental health survey dataset using the `mlcroissant` library, referencing all data entities by their Croissant `@id`. You can extend this notebook for deeper analyses, such as prevalence estimation, correlation between scores, or further demographic subgrouping.